# Camada Silver — Limpeza, Padronização e Transformações


## 0. Imports e Configurações

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import TimestampType, DoubleType, IntegerType

# Catalog e schemas
CATALOGO  = "workspace"
DB_BRONZE = "bronze"
DB_SILVER = "silver"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOGO}.{DB_SILVER}")
print(f"Banco de dados '{CATALOGO}.{DB_SILVER}' criado (ou já existia).")

## 1. Leitura das Tabelas Bronze

In [0]:
customers_bronze_df  = spark.table(f"{CATALOGO}.{DB_BRONZE}.tb_customers")
orders_bronze_df     = spark.table(f"{CATALOGO}.{DB_BRONZE}.tb_orders")
items_bronze_df      = spark.table(f"{CATALOGO}.{DB_BRONZE}.tb_order_items")
payments_bronze_df   = spark.table(f"{CATALOGO}.{DB_BRONZE}.tb_order_payments")
reviews_bronze_df    = spark.table(f"{CATALOGO}.{DB_BRONZE}.tb_order_reviews")
products_bronze_df   = spark.table(f"{CATALOGO}.{DB_BRONZE}.tb_products")
sellers_bronze_df    = spark.table(f"{CATALOGO}.{DB_BRONZE}.tb_sellers")
traducao_bronze_df   = spark.table(f"{CATALOGO}.{DB_BRONZE}.tb_product_category_name_translation")
cotacao_bronze_df    = spark.table(f"{CATALOGO}.{DB_BRONZE}.tb_cotacao_dolar")

print("Tabelas Bronze carregadas com sucesso!\n")

---
## 2. `silver.dim_consumidores` ← `bronze.tb_customers`

In [0]:
# Data Profiling
customers_bronze_df.display()
customers_bronze_df.printSchema()

nulos_df = customers_bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in customers_bronze_df.columns
]).display()

In [0]:
w_consumidor = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

dim_consumidores_df = (
    customers_bronze_df
    .withColumn("rn", F.row_number().over(w_consumidor))
    .filter(F.col("rn") == 1)
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_unique_id").alias("nome_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("customer_city")).alias("cidade"),
        F.upper(F.col("customer_state")).alias("estado"),
    )
    .filter(F.col("id_consumidor").isNotNull())
    .withColumn("data_ingestao", F.current_timestamp())
)

dim_consumidores_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{DB_SILVER}.dim_consumidores")
print(f"✅ Tabela silver.dim_consumidores criada com sucesso! ({dim_consumidores_df.count()} registros)\n")

---
## 3. `silver.fat_pedidos` ← `bronze.tb_orders`

In [0]:
orders_bronze_df.display()
orders_bronze_df.printSchema()

nulos_df = orders_bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in orders_bronze_df.columns
]).display()

In [0]:
mapa_status = {
    "delivered"  : "entregue",
    "canceled"   : "cancelado",
    "shipped"    : "enviado",
    "processing" : "processando",
    "invoiced"   : "faturado",
    "unavailable": "indisponível",
    "created"    : "criado",
    "approved"   : "aprovado",
}

expr_status = F.col("order_status")
for en, pt in mapa_status.items():
    expr_status = F.when(F.col("order_status") == en, pt).otherwise(expr_status)

fat_pedidos_df = (
    orders_bronze_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),
        expr_status.alias("status"),
        F.to_timestamp("order_purchase_timestamp").alias("data_compra"),
        F.to_timestamp("order_approved_at").alias("data_aprovacao"),
        F.to_timestamp("order_delivered_carrier_date").alias("data_entrega_transportadora"),
        F.to_timestamp("order_delivered_customer_date").alias("data_entrega_cliente"),
        F.to_timestamp("order_estimated_delivery_date").alias("data_entrega_estimada"),
    )
    .filter(F.col("id_pedido").isNotNull())
    .withColumn("tempo_entrega_dias",
        F.datediff(F.col("data_entrega_cliente"), F.col("data_compra")))
    .withColumn("tempo_entrega_estimado_dias",
        F.datediff(F.col("data_entrega_estimada"), F.col("data_compra")))
    .withColumn("diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias"))
    .withColumn("entrega_no_prazo",
        F.when(F.col("status") != "entregue",        "Não Entregue")
         .when(F.col("diferenca_entrega_dias") <= 0,  "Sim")
         .otherwise(                                   "Não"))
    .withColumn("data_ingestao", F.current_timestamp())
)

fat_pedidos_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{DB_SILVER}.fat_pedidos")
print(f"✅ Tabela silver.fat_pedidos criada com sucesso! ({fat_pedidos_df.count()} registros)\n")

---
## 4. `silver.fat_itens_pedidos` ← `bronze.tb_order_items`

In [0]:
items_bronze_df.display()
items_bronze_df.printSchema()

In [0]:
fat_itens_df = (
    items_bronze_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("order_item_id").alias("id_item"),
        F.col("product_id").alias("id_produto"),
        F.col("seller_id").alias("id_vendedor"),
        F.col("price").cast("double").alias("preco_BRL"),
        F.col("freight_value").cast("double").alias("preco_frete"),
    )
    .filter(F.col("id_pedido").isNotNull())
    .withColumn("data_ingestao", F.current_timestamp())
)

# shipping_limit_date removida — não utilizada nas análises da camada Gold

fat_itens_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{DB_SILVER}.fat_itens_pedidos")
print(f"✅ Tabela silver.fat_itens_pedidos criada com sucesso! ({fat_itens_df.count()} registros)\n")

---
## 5. `silver.fat_pagamentos_pedidos` ← `bronze.tb_order_payments`

In [0]:
payments_bronze_df.display()
payments_bronze_df.printSchema()

In [0]:
mapa_pagamento = {
    "credit_card" : "Cartão de Crédito",
    "boleto"      : "Boleto",
    "voucher"     : "Voucher",
    "debit_card"  : "Cartão de Débito",
    "not_defined" : "Não Definido",
}

expr_pag = F.col("payment_type")
for en, pt in mapa_pagamento.items():
    expr_pag = F.when(F.col("payment_type") == en, pt).otherwise(expr_pag)

fat_pagamentos_df = (
    payments_bronze_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("sequencial_pagamento"),
        expr_pag.alias("tipo_pagamento"),
        F.col("payment_installments").cast("int").alias("parcelas"),
        F.col("payment_value").cast("double").alias("valor_pagamento"),
    )
    .filter(F.col("id_pedido").isNotNull())
    .withColumn("data_ingestao", F.current_timestamp())
)

fat_pagamentos_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{DB_SILVER}.fat_pagamentos_pedidos")
print(f"✅ Tabela silver.fat_pagamentos_pedidos criada com sucesso! ({fat_pagamentos_df.count()} registros)\n")

---
## 6. `silver.fat_avaliacoes_pedidos` ← `bronze.tb_order_reviews`

In [0]:
reviews_bronze_df.display()
reviews_bronze_df.printSchema()

nulos_df = reviews_bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in reviews_bronze_df.columns
]).display()

In [0]:
fat_avaliacoes_df = (
    reviews_bronze_df
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        # try_cast evita erro quando a coluna tem valor inesperado no lugar do número
        F.expr("try_cast(review_score as int)").alias("nota_avaliacao"),
        F.coalesce(F.trim(F.col("review_comment_title")),   F.lit("Sem título")).alias("comentario_avaliacao"),
        F.coalesce(F.trim(F.col("review_comment_message")), F.lit("Sem comentário")).alias("mensagem_avaliacao"),
        F.expr("try_to_timestamp(review_creation_date)").alias("data_criacao_avaliacao"),
        F.expr("try_to_timestamp(review_answer_timestamp)").alias("data_resposta_avaliacao"),
    )
    .filter(F.col("id_pedido").isNotNull())
    .filter(
        F.col("data_criacao_avaliacao").isNull() |
        (F.col("data_criacao_avaliacao") <= F.current_timestamp())
    )
    .withColumn("data_ingestao", F.current_timestamp())
)

fat_avaliacoes_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{DB_SILVER}.fat_avaliacoes_pedidos")
print(f"✅ Tabela silver.fat_avaliacoes_pedidos criada com sucesso! ({fat_avaliacoes_df.count()} registros)\n")

---
## 7. `silver.dim_produtos` ← `bronze.tb_products`

In [0]:
products_bronze_df.display()
products_bronze_df.printSchema()

In [0]:
w_produto = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())

dim_produtos_df = (
    products_bronze_df
    .withColumn("rn", F.row_number().over(w_produto))
    .filter(F.col("rn") == 1)
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_category_name").alias("categoria_produto"),
        F.col("product_name_lenght").cast("int").alias("tamanho_nome_produto"),
        F.col("product_description_lenght").cast("int").alias("tamanho_descricao_produto"),
        F.col("product_photos_qty").cast("int").alias("quantidade_fotos"),
        F.col("product_weight_g").cast("double").alias("peso_produto_gramas"),
        F.col("product_length_cm").cast("double").alias("comprimento_centimetros"),
        F.col("product_height_cm").cast("double").alias("altura_centimetros"),
        F.col("product_width_cm").cast("double").alias("largura_centimetros"),
    )
    .filter(F.col("id_produto").isNotNull())
    .withColumn("data_ingestao", F.current_timestamp())
)

# product_name não existe no dataset Olist — identificação feita pela categoria

dim_produtos_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{DB_SILVER}.dim_produtos")
print(f"✅ Tabela silver.dim_produtos criada com sucesso! ({dim_produtos_df.count()} registros)\n")

---
## 8. `silver.dim_vendedores` ← `bronze.tb_sellers`

In [0]:
sellers_bronze_df.display()
sellers_bronze_df.printSchema()

In [0]:
w_vendedor = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

dim_vendedores_df = (
    sellers_bronze_df
    .withColumn("rn", F.row_number().over(w_vendedor))
    .filter(F.col("rn") == 1)
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("seller_city")).alias("cidade"),
        F.upper(F.col("seller_state")).alias("estado"),
    )
    .filter(F.col("id_vendedor").isNotNull())
    .withColumn("data_ingestao", F.current_timestamp())
)

# seller_name não existe no dataset Olist — identificação feita pelo seller_id

dim_vendedores_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{DB_SILVER}.dim_vendedores")
print(f"✅ Tabela silver.dim_vendedores criada com sucesso! ({dim_vendedores_df.count()} registros)\n")

---
## 9. `silver.dim_categoria_produtos_traducao` ← `bronze.tb_product_category_name_translation`

In [0]:
traducao_bronze_df.display()
traducao_bronze_df.printSchema()

In [0]:
dim_traducao_df = (
    traducao_bronze_df
    .select(
        F.col("product_category_name").alias("nome_produto_pt"),
        F.col("product_category_name_english").alias("nome_produto_en"),
    )
    .filter(F.col("nome_produto_pt").isNotNull())
    .withColumn("data_ingestao", F.current_timestamp())
)

dim_traducao_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{DB_SILVER}.dim_categoria_produtos_traducao")
print(f"✅ Tabela silver.dim_categoria_produtos_traducao criada com sucesso! ({dim_traducao_df.count()} registros)\n")

---
## 10. `silver.dim_cotacao_dolar` — Calendário Contínuo

In [0]:
cotacao_bronze_df.display()
cotacao_bronze_df.printSchema()

In [0]:
df_cotacao = (
    cotacao_bronze_df
    .select(
        F.to_date(F.to_timestamp("dataHoraCotacao")).alias("data_cotacao"),
        F.col("cotacaoCompra").cast("double").alias("cotacao_compra"),
    )
    .filter(F.col("data_cotacao").isNotNull())
    .orderBy("data_cotacao")
)

data_min = df_cotacao.agg(F.min("data_cotacao")).collect()[0][0]
data_max = df_cotacao.agg(F.max("data_cotacao")).collect()[0][0]

df_calendario = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{data_min}'),
        to_date('{data_max}'),
        interval 1 day
    )) AS data_cotacao
""")

df_join = df_calendario.join(df_cotacao, on="data_cotacao", how="left")

w_fill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

dim_cotacao_df = (
    df_join
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(w_fill))
    .withColumn("data_ingestao", F.current_timestamp())
)

dim_cotacao_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{DB_SILVER}.dim_cotacao_dolar")
print(f"✅ Tabela silver.dim_cotacao_dolar criada com sucesso! ({dim_cotacao_df.count()} registros)\n")

---
## 11. `silver.fat_pedido_total` — Tabela Final Integrada

In [0]:
fat_pedidos_s    = spark.table(f"{CATALOGO}.{DB_SILVER}.fat_pedidos")
fat_pagamentos_s = spark.table(f"{CATALOGO}.{DB_SILVER}.fat_pagamentos_pedidos")
dim_cotacao_s    = spark.table(f"{CATALOGO}.{DB_SILVER}.dim_cotacao_dolar")

pag_agg_df = (
    fat_pagamentos_s
    .groupBy("id_pedido")
    .agg(F.round(F.sum("valor_pagamento"), 2).alias("valor_total_pago_brl"))
)

fat_pedido_total_df = (
    fat_pedidos_s
    .join(pag_agg_df, on="id_pedido", how="left")
    .join(
        dim_cotacao_s.select("data_cotacao", "cotacao_compra"),
        F.to_date(fat_pedidos_s["data_compra"]) == dim_cotacao_s["data_cotacao"],
        how="left"
    )
    .withColumn("valor_total_pago_usd",
        F.round(F.col("valor_total_pago_brl") / F.col("cotacao_compra"), 2))
    .withColumn("valor_total_pago_brl",
        F.round(F.col("valor_total_pago_brl"), 2))
    .select(
        "id_pedido",
        "id_consumidor",
        "status",
        "valor_total_pago_brl",
        "valor_total_pago_usd",
        F.to_date("data_compra").alias("data_pedido"),
    )
    .withColumn("data_ingestao", F.current_timestamp())
)

fat_pedido_total_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{DB_SILVER}.fat_pedido_total")
print(f"✅ Tabela silver.fat_pedido_total criada com sucesso! ({fat_pedido_total_df.count()} registros)\n")

## 12. Otimização Delta Lake (OPTIMIZE + ZORDER)

In [0]:
tabelas_otimizar = [
    (f"{CATALOGO}.{DB_SILVER}.fat_pedidos",            "id_pedido, data_compra"),
    (f"{CATALOGO}.{DB_SILVER}.fat_pedido_total",       "id_pedido, data_pedido"),
    (f"{CATALOGO}.{DB_SILVER}.fat_itens_pedidos",      "id_pedido"),
    (f"{CATALOGO}.{DB_SILVER}.fat_pagamentos_pedidos", "id_pedido"),
    (f"{CATALOGO}.{DB_SILVER}.fat_avaliacoes_pedidos", "id_pedido"),
]

for tabela, colunas in tabelas_otimizar:
    print(f"Otimizando {tabela}...")
    spark.sql(f"OPTIMIZE {tabela} ZORDER BY ({colunas})")
    print(f"  ✅ {tabela} otimizada.")

## 13. Verificação Final

In [0]:
print("=" * 60)
print("TABELAS CRIADAS NA CAMADA SILVER")
print("=" * 60)
spark.sql(f"SHOW TABLES IN {CATALOGO}.{DB_SILVER}").show(truncate=False)

print("\nProcesso da camada Silver concluído com sucesso!")